Qwen3-VL 8B Instruct + Unsloth + optional MediaPipe pose

**Notebook vs local scripts:** You do **not** have to use a notebook. The same flow runs as plain Python:

- `train_qwen3_vl_8b.py` / `infer_qwen3_vl_8b.py` with `--pose_mode` etc.

Use this notebook when you want **interactive** runs (Colab, Jupyter, or VS Code). Use the scripts for **automation**, CI, or headless servers.

**GPU:** Training expects **CUDA** (Unsloth + 4-bit). Colab T4 works; A100 / Ampere uses bf16 automatically in the train cell; local Linux + NVIDIA works too.

### Colab: upload `backend` from your Mac

From your machine (repo root), zip **only** `backend` (excludes heavy `venv`):

```bash
cd /path/to/IsoCourt && zip -r backend.zip backend -x "backend/venv/*" -x "backend/**/__pycache__/*" -x "backend/**/*.pyc"
```

In Colab: **Upload** `backend.zip`, then in a cell:

```python
!unzip -q -o backend.zip -d /content
```

You should get `/content/backend/pipelines/vlm/common/load_dataset_jsonl.py`.

### Paths and imports

The next cell finds `backend/pipelines/vlm` (with `common/` + `qwen-8b/`) whether you uploaded **only `backend`** (`/content/backend/...`) or the **full repo** (`/content/IsoCourt/backend/...`). You can override with `VLM_DIR = Path("...")` or env `ISOCOURT_VLM_DIR`.

In [ ]:
import os
import sys
from pathlib import Path
from typing import Optional

_VLM_MARKER = "load_dataset_jsonl.py"


def _is_vlm_root(p: Path) -> bool:
    return p.is_dir() and (p / "common" / _VLM_MARKER).is_file()


def _vlm_under_backend_root(backend_root: Path) -> Optional[Path]:
    cand = (backend_root / "pipelines" / "vlm").resolve()
    return cand if _is_vlm_root(cand) else None


def _vlm_under_repo_root(repo_root: Path) -> Optional[Path]:
    cand = (repo_root / "backend" / "pipelines" / "vlm").resolve()
    return cand if _is_vlm_root(cand) else None


def _search_bases(cwd: Path):
    seen: set[Path] = set()
    for b in _iter_search_roots(cwd):
        b = b.resolve()
        if b not in seen:
            seen.add(b)
            yield b


def _iter_search_roots(cwd: Path):
    yield cwd
    yield cwd / "IsoCourt"
    content = Path("/content")
    if content.is_dir():
        yield content / "backend"
        try:
            for child in content.iterdir():
                if child.is_dir():
                    yield child
        except OSError:
            pass
    try:
        for child in cwd.iterdir():
            if child.is_dir():
                yield child
    except OSError:
        pass
    drive = cwd / "drive" / "MyDrive"
    if drive.is_dir():
        try:
            for a in drive.iterdir():
                if not a.is_dir():
                    continue
                yield a
                try:
                    for b in a.iterdir():
                        if b.is_dir():
                            yield b
                except OSError:
                    pass
        except OSError:
            pass


def _resolve_vlm_root() -> Path:
    env = os.environ.get("ISOCOURT_VLM_DIR") or os.environ.get("VLM_DIR")
    if env:
        pth = Path(env).expanduser().resolve()
        if _is_vlm_root(pth):
            return pth

    cwd = Path.cwd().resolve()
    if _is_vlm_root(cwd):
        return cwd

    for base in _search_bases(cwd):
        hit = _vlm_under_backend_root(base)
        if hit is not None:
            return hit
        hit = _vlm_under_repo_root(base)
        if hit is not None:
            return hit

    for parent in cwd.parents:
        hit = _vlm_under_backend_root(parent)
        if hit is not None:
            return hit
        hit = _vlm_under_repo_root(parent)
        if hit is not None:
            return hit

    return cwd


# Optional: force path if auto-detect fails (Colab after unzip to /content):
# VLM_ROOT = Path("/content/backend/pipelines/vlm")

VLM_ROOT = _resolve_vlm_root()
QWEN_DIR = VLM_ROOT / "qwen-8b"
BACKEND_ROOT = VLM_ROOT.parent.parent
for pth in (VLM_ROOT / "common", QWEN_DIR, BACKEND_ROOT):
    if str(pth) not in sys.path:
        sys.path.insert(0, str(pth))

if not _is_vlm_root(VLM_ROOT):
    raise FileNotFoundError(
        f"Could not find common/{_VLM_MARKER}. Tried backend-only layout (/content/backend/pipelines/vlm) "
        f"and full repo (/content/…/backend/pipelines/vlm). cwd={Path.cwd()!s}. "
        f"Set VLM_DIR = Path(\"/content/backend/pipelines/vlm\") or env ISOCOURT_VLM_DIR."
    )

print("VLM_ROOT:", VLM_ROOT)
print("QWEN_DIR:", QWEN_DIR)
print("BACKEND_ROOT:", BACKEND_ROOT)


### Install dependencies

**Local:** `pip install -r requirements-unsloth-vlm.txt` in your venv (from `backend/pipelines/vlm/common`).

**Colab:** uncomment and run the pip cell below once per runtime.

In [ ]:
# Run once per fresh runtime.
# If this notebook kernel already has deps installed, you can skip.
%pip install -U pip
%pip install unsloth
%pip install "transformers>=4.57.0" "trl>=0.22.0" datasets accelerate bitsandbytes peft sentencepiece protobuf pillow mediapipe opencv-python

# Optional HF login if model pulls fail with auth errors:
# from huggingface_hub import login
# login()

### Configuration

**Pose landmarker:** Download `pose_landmarker_lite.task` from [MediaPipe pose landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker) and set `POSE_MODEL_PATH` to the file, or place it at `backend/models/pose_landmarker_lite.task` (same as `PoseEstimator` default).

In [ ]:
from qwen3_vl_config import (
    DEFAULT_MODEL_ID,
    DEFAULT_TRAIN_MAX_PIXELS,
    DEFAULT_TRAIN_MAX_SEQ_LENGTH,
)

MODEL_NAME = DEFAULT_MODEL_ID  # unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit
OUTPUT_DIR = QWEN_DIR / "outputs" / "notebook_qwen3vl8b"
# Regenerate: python build_finebadminton_jsonl.py (from this folder)
JSONL_PATH = BACKEND_ROOT / "data" / "FineBadminton-master" / "dataset" / "finebadminton_vlm_train.jsonl"

# MediaPipe: none | overlay | text | both
POSE_MODE = "both"
POSE_MODEL_PATH = BACKEND_ROOT / "models" / "pose_landmarker_lite.task"
# Upscale before MediaPipe if min(h,w) < this (720p wide shots → ~1.33×). Set 0 to disable.
POSE_MIN_SHORT_EDGE = 960

# Training default 1536 (see qwen3_vl_config); use 2048 from qwen3_vl_config if captions are long.
MAX_SEQ_LENGTH = DEFAULT_TRAIN_MAX_SEQ_LENGTH
MAX_STEPS = None  # set e.g. 30 for a quick smoke test; None = use NUM_EPOCHS only
NUM_EPOCHS = 25
# Colab: stroke metrics need logits on CPU — None = full val if RAM allows.
MAX_EVAL_SAMPLES = 32  # None = full val set
# Align vision preprocessor with PIL (0 = skip patch; uses HF defaults).
MAX_PIXELS = DEFAULT_TRAIN_MAX_PIXELS
MIN_PIXELS = None  # optional Qwen smart_resize floor
SAVE_STEPS = None  # e.g. 100 for extra step checkpoints; None = epoch saves only
# Downscale frames before pose/VLM (None = full resolution). 1280 helps 12GB Colab CPU RAM.
MAX_IMAGE_LONG_EDGE = 1280  # set None to disable
# Best checkpoint uses eval_stroke_accuracy (needs logits on CPU). If eval OOMs: False or lower MAX_EVAL_SAMPLES.
EVAL_STROKE_METRICS = True
# LoRA init, HF Trainer, and train/val split (passed through load_jsonl_conversations_train_val).
SEED = 42

assert JSONL_PATH.exists(), f"JSONL not found: {JSONL_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    POSE_MIN_SHORT_EDGE
except NameError:
    POSE_MIN_SHORT_EDGE = 960  # add above if you edit this cell: # POSE_MIN_SHORT_EDGE = 960

try:
    MAX_IMAGE_LONG_EDGE
except NameError:
    MAX_IMAGE_LONG_EDGE = 1280

try:
    EVAL_STROKE_METRICS
except NameError:
    EVAL_STROKE_METRICS = True

try:
    SEED
except NameError:
    SEED = 42

try:
    MAX_PIXELS
except NameError:
    MAX_PIXELS = DEFAULT_TRAIN_MAX_PIXELS

print("JSONL_PATH:", JSONL_PATH)
print("POSE_MODE:", POSE_MODE)
print("POSE_MODEL_PATH:", POSE_MODEL_PATH)
print("POSE_MIN_SHORT_EDGE:", POSE_MIN_SHORT_EDGE)
print("MAX_EVAL_SAMPLES:", MAX_EVAL_SAMPLES)
print("MAX_IMAGE_LONG_EDGE:", MAX_IMAGE_LONG_EDGE)
print("EVAL_STROKE_METRICS:", EVAL_STROKE_METRICS)
print("SEED:", SEED)
print("MAX_PIXELS:", MAX_PIXELS)

In [ ]:
# Preflight checks: GPU + JSONL format + optional pose task file
import json
import random

import numpy as np
import torch

try:
    SEED
except NameError:
    SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No CUDA GPU. On Colab, switch Runtime -> GPU.")

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()
row = json.loads(first_line)
for key in ("image", "instruction", "response"):
    assert key in row, f"Missing key '{key}' in JSONL row"
print("JSONL keys OK")

if POSE_MODE != "none":
    if not POSE_MODEL_PATH.exists():
        print("Pose model missing. Downloading pose_landmarker_lite.task...")
        import urllib.request
        url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
        POSE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, POSE_MODEL_PATH)
    print("Pose model path:", POSE_MODEL_PATH)

### Load model + LoRA

In [ ]:
import torch

try:
    SEED
except NameError:
    SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for Unsloth Qwen3-VL training in this notebook.")

from unsloth import FastVisionModel

from vlm_processor_utils import apply_vision_processor_limits

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

try:
    MAX_PIXELS
except NameError:
    from qwen3_vl_config import DEFAULT_TRAIN_MAX_PIXELS

    MAX_PIXELS = DEFAULT_TRAIN_MAX_PIXELS
try:
    MIN_PIXELS
except NameError:
    MIN_PIXELS = None
_mp = None if MAX_PIXELS == 0 else MAX_PIXELS
apply_vision_processor_limits(tokenizer, max_pixels=_mp, min_pixels=MIN_PIXELS)
if _mp is not None:
    print("Vision preprocessor max_pixels:", _mp)

### Build training set (JSONL + optional MediaPipe pose)

Images are **lazy-loaded** per batch (paths + JSON rows stay in RAM, not every PIL frame). Same API as `load_jsonl_conversations_train_val(...)` in `load_dataset_jsonl.py`.

In [ ]:
import importlib

# Colab: editing .py on disk does NOT refresh imports — reload (or Runtime → Restart session).
import vlm_pose
importlib.reload(vlm_pose)
import load_dataset_jsonl
importlib.reload(load_dataset_jsonl)
from torch.utils.data import Subset

from load_dataset_jsonl import load_jsonl_conversations_train_val, trainer_vision_kwargs

try:
    SEED
except NameError:
    SEED = 42

_pose_min = None if POSE_MIN_SHORT_EDGE == 0 else POSE_MIN_SHORT_EDGE
train_dataset, val_dataset = load_jsonl_conversations_train_val(
    str(JSONL_PATH),
    pose_mode=POSE_MODE,
    pose_model_path=str(POSE_MODEL_PATH) if POSE_MODEL_PATH else None,
    pose_min_short_edge=_pose_min,
    max_image_long_edge=MAX_IMAGE_LONG_EDGE,
    split_seed=SEED,
)
if MAX_EVAL_SAMPLES is not None and len(val_dataset) > MAX_EVAL_SAMPLES:
    val_dataset = Subset(val_dataset, range(MAX_EVAL_SAMPLES))
len(train_dataset), len(val_dataset)

### Train

In [ ]:
from vlm_gpu_defaults import default_mixed_precision_flags

from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

from vlm_train_metrics import build_sft_eval_compute_metrics

try:
    SEED
except NameError:
    SEED = 42

try:
    SAVE_STEPS
except NameError:
    SAVE_STEPS = None

FastVisionModel.for_training(model)

tkwargs = trainer_vision_kwargs(max_length=MAX_SEQ_LENGTH)
train_kwargs = dict(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    gradient_accumulation_steps=8,
    warmup_steps=5,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=SEED,
    output_dir=str(OUTPUT_DIR),
    report_to="none",
    bf16=default_mixed_precision_flags()[0],
    fp16=default_mixed_precision_flags()[1],
    eval_strategy="epoch",
    save_total_limit=5,
    load_best_model_at_end=True,
    **tkwargs,
)
if SAVE_STEPS is not None:
    train_kwargs["save_strategy"] = "steps"
    train_kwargs["save_steps"] = SAVE_STEPS
else:
    train_kwargs["save_strategy"] = "epoch"
if EVAL_STROKE_METRICS:
    train_kwargs["metric_for_best_model"] = "eval_stroke_accuracy"
    train_kwargs["greater_is_better"] = True
    _eval_metrics = build_sft_eval_compute_metrics(tokenizer)
else:
    train_kwargs["metric_for_best_model"] = "eval_loss"
    train_kwargs["greater_is_better"] = False
    train_kwargs["prediction_loss_only"] = True
    _eval_metrics = None
if MAX_STEPS is not None:
    train_kwargs["max_steps"] = MAX_STEPS
else:
    train_kwargs["num_train_epochs"] = NUM_EPOCHS

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=_eval_metrics,
    args=SFTConfig(**train_kwargs),
)

trainer.train()

### Save LoRA adapter

In [ ]:
adapter_dir = OUTPUT_DIR / "lora_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved:", adapter_dir)

### Inference (optional)

Same behavior as `infer_qwen3_vl_8b.py`: optional pose overlay/text before the VLM.

In [ ]:
from PIL import Image
from transformers import TextStreamer

from vlm_pose import apply_pose_to_pil, create_pose_estimator

_fb = BACKEND_ROOT / "data" / "FineBadminton-master" / "dataset" / "image" / "0011_001_16363.jpg"
IMAGE_PATH = _fb if _fb.is_file() else (VLM_ROOT / "common" / "example_data" / "sample.jpg")
USER_PROMPT = "Describe this image."
INFER_POSE_MODE = "none"  # or overlay / text / both

image = Image.open(IMAGE_PATH).convert("RGB")
prompt = USER_PROMPT
if INFER_POSE_MODE != "none":
    pe = create_pose_estimator(str(POSE_MODEL_PATH) if POSE_MODEL_PATH else None)
    _pm = None if POSE_MIN_SHORT_EDGE == 0 else POSE_MIN_SHORT_EDGE
    image, prompt = apply_pose_to_pil(
        image, pe, mode=INFER_POSE_MODE, instruction=prompt, min_short_edge_for_pose=_pm
    )

FastVisionModel.for_inference(model)

messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
)
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)